Strategic Analysis

In [2]:
print("Rank products:")
print("-" * 50)

# Query to rank products by total sales
query_product_rank = """
    WITH ProductSales AS (
        SELECT
            p.productName,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            products p
        JOIN
            orderdetails od ON p.productCode = od.productCode
        GROUP BY
            p.productName
    )
    SELECT
        productName,
        totalSales,
        -- RANK() assigns a rank to each product based on total sales
        RANK() OVER (ORDER BY totalSales DESC) AS salesRank
    FROM
        ProductSales
    ORDER BY
        salesRank;
"""

df_product_rank = pd.read_sql(query_product_rank, conn)
print(df_product_rank)

Rank products:
--------------------------------------------------
                             productName  totalSales  salesRank
0            1992 Ferrari 360 Spider red   239241.31          1
1                      2001 Ferrari Enzo   176710.58          2
2               1952 Alpine Renault 1300   170533.93          3
3                      1968 Ford Mustang   144121.12          4
4                       1969 Ford Falcon   143597.82          5
..                                   ...         ...        ...
104                    1982 Ducati 996 R    29020.71        105
105     1936 Mercedes Benz 500k Roadster    28618.59        106
106              1982 Lamborghini Diablo    27407.32        107
107  1958 Chevy Corvette Limited Edition    26698.98        108
108          1939 Chevrolet Deluxe Coupe    24027.56        109

[109 rows x 3 columns]


Product Trends

In [3]:
print("month-over-month sales trends for a specific product:")
print("-" * 50)

# Query to analyze month-over-month sales trends for '1992 Ferrari 360 Spider red'
query_sales_trends = """
    WITH MonthlyProductSales AS (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    )
    SELECT
        salesMonth,
        totalSales,
        -- LAG() gets the sales from the previous month
        LAG(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS previousMonthSales,
        -- LEAD() gets the sales from the next month
        LEAD(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS nextMonthSales
    FROM
        MonthlyProductSales
    ORDER BY
        salesMonth;
"""

df_sales_trends = pd.read_sql(query_sales_trends, conn)
print(df_sales_trends)

month-over-month sales trends for a specific product:
--------------------------------------------------
   salesMonth  totalSales  previousMonthSales  nextMonthSales
0     2018-01     3816.85                0.00         7400.02
1     2018-03     7400.02             3816.85         8128.32
2     2018-04     8128.32             7400.02         3429.25
3     2018-05     3429.25             8128.32         3278.44
4     2018-06     3278.44             3429.25         6942.94
5     2018-07     6942.94             3278.44         4893.96
6     2018-08     4893.96             6942.94         8126.73
7     2018-09     8126.73             4893.96        13169.51
8     2018-10    13169.51             8126.73        33221.01
9     2018-11    33221.01            13169.51        11073.27
10    2018-12    11073.27            33221.01         6231.60
11    2019-01     6231.60            11073.27        12172.45
12    2019-02    12172.45             6231.60         3464.78
13    2019-03     3464.78  

Employee Performance

In [4]:
print("Rank employees:")
print("-" * 50)

# Query to rank employees by total sales
query_employee_rank = """
    WITH EmployeeSales AS (
        SELECT
            e.firstName || ' ' || e.lastName AS employeeName,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            employees e
        JOIN
            customers c ON e.employeeNumber = c.salesRepEmployeeNumber
        JOIN
            orders o ON c.customerNumber = o.customerNumber
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        GROUP BY
            employeeName
    )
    SELECT
        employeeName,
        totalSales,
        -- RANK() assigns a rank to each employee based on total sales
        RANK() OVER (ORDER BY totalSales DESC) AS salesRank
    FROM
        EmployeeSales
    ORDER BY
        salesRank;
"""

df_employee_rank = pd.read_sql(query_employee_rank, conn)
print(df_employee_rank)

Rank employees:
--------------------------------------------------
        employeeName  totalSales  salesRank
0   Gerard Hernandez   962660.38          1
1    Leslie Jennings   884247.13          2
2    Pamela Castillo   741394.75          3
3         Larry Bott   694837.85          4
4        Barry Jones   676887.37          5
5      George Vanauf   597351.23          6
6        Loui Bondur   492709.87          7
7        Andy Fixter   479537.30          8
8     Foon Yue Tseng   459142.29          9
9         Mami Nishi   452978.58         10
10   Steve Patterson   418925.36         11
11       Peter Marsh   416923.92         12
12     Martin Gerard   387477.47         13
13    Julie Firrelli   386663.20         14
14   Leslie Thompson   347533.03         15


Detailed Product Analysis: '1992 Ferrari 360 Spider red'

In [6]:
print("Detailed Analysis of '1992 Ferrari 360 Spider red' Product Performance:")
print("-" * 50)

# ----- Monthly Sales Rank (RANK) -----
print("--- 1. Monthly Sales Rank (RANK) ---")
query_rank = """
    SELECT
        strftime('%Y-%m', o.orderDate) AS salesMonth,
        SUM(od.quantityOrdered * od.priceEach) AS totalSales,
        RANK() OVER (ORDER BY SUM(od.quantityOrdered * od.priceEach) DESC) AS salesRank
    FROM
        orders o
    JOIN
        orderdetails od ON o.orderNumber = od.orderNumber
    JOIN
        products p ON p.productCode = od.productCode
    WHERE
        p.productName = '1992 Ferrari 360 Spider red'
    GROUP BY
        salesMonth;
"""
df_rank = pd.read_sql(query_rank, conn)
print(df_rank)
print("\n" + "="*80 + "\n")

# ----- Sales Comparison with Previous and Next Month (LAG/LEAD) -----
print("--- 2. Sales Comparison ---")
query_lag_lead = """
    SELECT
        salesMonth,
        totalSales,
        LAG(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS previousMonthSales,
        LEAD(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS nextMonthSales
    FROM (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    ) AS monthly_sales;
"""
df_lag_lead = pd.read_sql(query_lag_lead, conn)
print(df_lag_lead)
print("\n" + "="*80 + "\n")

# ----- Sales Division into Quartiles for Performance (NTILE) -----
print("--- 3. Sales Division into Quartiles ---")
query_ntile = """
    SELECT
        salesMonth,
        totalSales,
        NTILE(4) OVER (ORDER BY totalSales) AS salesQuartile
    FROM (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    ) AS monthly_sales;
"""
df_ntile = pd.read_sql(query_ntile, conn)
print(df_ntile)
print("\n" + "="*80 + "\n")

Detailed Analysis of '1992 Ferrari 360 Spider red' Product Performance:
--------------------------------------------------
--- 1. Monthly Sales Rank (RANK) ---
   salesMonth  totalSales  salesRank
0     2018-11    33221.01          1
1     2019-11    21672.45          2
2     2019-10    17191.33          3
3     2019-08    15366.17          4
4     2018-10    13169.51          5
5     2019-12    12273.92          6
6     2019-02    12172.45          7
7     2018-12    11073.27          8
8     2019-06     9940.27          9
9     2020-01     9765.95         10
10    2018-04     8128.32         11
11    2018-09     8126.73         12
12    2018-03     7400.02         13
13    2019-07     7364.73         14
14    2018-07     6942.94         15
15    2019-04     6367.09         16
16    2019-01     6231.60         17
17    2020-02     5613.66         18
18    2019-05     5242.68         19
19    2018-08     4893.96         20
20    2018-01     3816.85         21
21    2019-03     3464.78 

Organizational Mapping

In [7]:
# Set the employee ID you want to start the hierarchy from
employee_id = 1056

print(f"--- Command Hierarchy for Employee ID: {employee_id} ---")
print("-" * 50)

# Recursive query to find the complete employee hierarchy
query_recursive = f"""
    WITH RECURSIVE EmployeeHierarchy AS (
        -- Base case: Selects the initial employee to start from
        SELECT
            employeeNumber,
            reportsTo,
            firstName || ' ' || lastName AS employeeName,
            1 AS level
        FROM
            employees
        WHERE
            employeeNumber = {employee_id}

        UNION ALL

        -- Recursive case: Joins to itself to find the manager of the previous employee
        SELECT
            e.employeeNumber,
            e.reportsTo,
            e.firstName || ' ' || e.lastName AS employeeName,
            h.level + 1 AS level
        FROM
            employees e
        JOIN
            EmployeeHierarchy h ON e.employeeNumber = h.reportsTo
    )
    SELECT
        level,
        employeeName AS subordinate,
        (SELECT firstName || ' ' || lastName FROM employees WHERE employeeNumber = h.reportsTo) AS managerName
    FROM
        EmployeeHierarchy h
    ORDER BY
        level DESC;
"""

df_recursive = pd.read_sql(query_recursive, conn)
print(df_recursive)

--- Command Hierarchy for Employee ID: 1056 ---
--------------------------------------------------
   level     subordinate   managerName
0      2    Diane Murphy          None
1      1  Mary Patterson  Diane Murphy
